# Rebuild Hippo inputs from the eight regenerated sublibraries

Run All on a high-memory Sherlock compute node in the `sc-dl` environment. The inputs are the eight
`/scratch/users/edtng/Hippo_timecourse/Hippo:eTN8_GEX_sub{1..8}/kb_all_main_raw/counts_filtered/adata.h5ad`
files. The older Oak combined H5AD is not used. This notebook rebuilds the combined query, recreates the
selected `top_specific_200` fixed-reference cNMF projection, validates both, updates the local path
overrides, and executes the current Figure 3 and Figure 4 notebooks.

Exact historical cell totals are now recorded as diagnostics rather than used as acceptance gates. The
workflow still enforces the eight expected inputs, unique cell/feature IDs, the production broad-QC mask,
the 54-sample mapping, required raw-count layers and guide matrices, raw-count integrity, fixed 50-program
reference structure, exact query/projection cell alignment, CLR reconstruction, marker availability, and
all requested Figure 3/4 conditions.


In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository checkout.")

REPO_ROOT = _find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("Repository:", REPO_ROOT)


In [ ]:
from datetime import datetime, timezone
import json
import tempfile
import pandas as pd
from IPython.display import display

from src.cc_perturbseq.figure_notebook import get_path, load_paths
from src.cc_perturbseq.hippo_rebuild import (
    EXPECTED_POST_QC_SHAPE, EXPECTED_PRE_QC_SHAPE,
    HISTORICAL_SAMPLE_COUNTS, preflight_sublibraries, rebuild_combined,
    validate_combined,
)
from src.cc_perturbseq.hippo_projection import (
    EXPECTED_QC_PASS, EXPECTED_QUERY_SHAPE, EXPECTED_SELECTED_GENES,
    project_top_specific_200, read_spectra, validate_projected,
)
from src.cc_perturbseq.hippo_workflow import (
    check_runtime, execute_figure_notebooks, run_main_preflight,
    validate_hippo_panel_inputs, write_hippo_overrides,
)

config_file, paths = load_paths(REPO_ROOT)
INPUT_ROOT = get_path(paths, "hippo_rebuild.input_root")
REBUILD_ROOT = get_path(paths, "hippo_rebuild.output_root")
SPECTRA_TPM = get_path(paths, "hippo_rebuild.gene_spectra_tpm")

# Exact historical cell totals are informative but no longer block reconstruction.
ENFORCE_EXACT_HISTORICAL_COUNTS = False
STRICT_REFERENCE = True
REBUILD_STALE_OUTPUTS = True
FORCE_REBUILD_ALL = False
COMPRESSION = None
RUN_FIGURE_NOTEBOOKS = True
RUN_FULL_MAIN_PREFLIGHT = True

QUERY_H5AD = REBUILD_ROOT / "Hippo_timecourse_combined.h5ad"
PROJECTED_H5AD = (
    REBUILD_ROOT / "perturbseq_cNMF_projection" / "reprojection_gene_set_comparison"
    / "Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad"
)
RUN_DIR = REBUILD_ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

def build_or_replace_stale(builder, *args, label, **kwargs):
    try:
        return builder(*args, overwrite=FORCE_REBUILD_ALL, **kwargs)
    except FileExistsError as exc:
        if FORCE_REBUILD_ALL or not REBUILD_STALE_OUTPUTS:
            raise
        print(f"{label} is stale or untrusted: {exc}\nRebuilding from current inputs.", flush=True)
        return builder(*args, overwrite=True, **kwargs)

def historical_count_diagnostics(validation, pre_qc_shape):
    observed_counts = {str(k): int(v) for k, v in validation["sample_counts"].items()}
    sample_differences = {}
    for sample in sorted(set(observed_counts) | set(HISTORICAL_SAMPLE_COUNTS)):
        observed = observed_counts.get(sample, 0)
        recorded = HISTORICAL_SAMPLE_COUNTS.get(sample, 0)
        if observed != recorded:
            sample_differences[sample] = {
                "observed": observed,
                "recorded": recorded,
                "delta": observed - recorded,
            }
    observed_shape = tuple(validation["shape"])
    return {
        "acceptance_role": "diagnostic_only",
        "exact_historical_match": (
            observed_shape == EXPECTED_POST_QC_SHAPE and not sample_differences
        ),
        "pre_qc_observed": list(pre_qc_shape or []),
        "pre_qc_recorded": list(EXPECTED_PRE_QC_SHAPE),
        "post_qc_observed": list(observed_shape),
        "post_qc_recorded": list(EXPECTED_POST_QC_SHAPE),
        "post_qc_cell_delta": observed_shape[0] - EXPECTED_POST_QC_SHAPE[0],
        "sample_count_differences": sample_differences,
    }

display({
    "config": str(config_file),
    "input_root": str(INPUT_ROOT),
    "combined_output": str(QUERY_H5AD),
    "projection_output": str(PROJECTED_H5AD),
    "run_dir": str(RUN_DIR),
    "exact_historical_counts_enforced": ENFORCE_EXACT_HISTORICAL_COUNTS,
})


In [ ]:
versions = check_runtime(include_figures=RUN_FIGURE_NOTEBOOKS)
inputs = preflight_sublibraries(INPUT_ROOT)
if [x["sublibID"] for x in inputs] != [f"s{i}" for i in range(1, 9)]:
    raise ValueError("Expected the ordered s1...s8 sublibraries.")
table = pd.DataFrame(inputs)
table["mtime_utc"] = pd.to_datetime(table["mtime_ns"], unit="ns", utc=True)
display(table[["sublibID", "path", "size_bytes", "mtime_utc"]])

# Keep the fixed reference strict even though regenerated query cell totals may differ slightly.
reference = read_spectra(SPECTRA_TPM, strict_historical=STRICT_REFERENCE)
print("Reference shape:", reference.shape)
del reference

REBUILD_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=REBUILD_ROOT) as probe:
    probe.write(b"write check")
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "settings.json").write_text(json.dumps({
    "inputs": inputs,
    "spectra_tpm": str(SPECTRA_TPM),
    "query_h5ad": str(QUERY_H5AD),
    "projected_h5ad": str(PROJECTED_H5AD),
    "exact_historical_counts_enforced": ENFORCE_EXACT_HISTORICAL_COUNTS,
    "strict_reference": STRICT_REFERENCE,
    "rebuild_stale_outputs": REBUILD_STALE_OUTPUTS,
    "force_rebuild_all": FORCE_REBUILD_ALL,
    "versions": versions,
}, indent=2) + "\n")

combined = build_or_replace_stale(
    rebuild_combined, INPUT_ROOT, QUERY_H5AD,
    label="Combined Hippo query",
    strict_historical=False,
    compression=COMPRESSION,
)
current_validation = validate_combined(QUERY_H5AD, strict_historical=False)
historical_comparison = historical_count_diagnostics(
    current_validation, combined.get("pre_qc_shape"),
)
if ENFORCE_EXACT_HISTORICAL_COUNTS and not historical_comparison["exact_historical_match"]:
    raise ValueError(
        "The regenerated cohort differs from the historical cell counts: "
        + json.dumps(historical_comparison, indent=2)
    )
if not historical_comparison["exact_historical_match"]:
    print(
        "Historical cell-count differences are diagnostic only; continuing after "
        "structural, QC, mapping, and raw-count validation.",
        flush=True,
    )

combined["validation_after_final_path"] = current_validation
combined["historical_comparison"] = historical_comparison
combined["acceptance_basis"] = (
    "structural schema, production broad QC, 54-sample mapping, required layers/obsm, "
    "and raw-count integrity; historical cell totals are diagnostic only"
)
(RUN_DIR / "combined_stage.json").write_text(json.dumps(combined, indent=2) + "\n")
display(current_validation)
display(historical_comparison)


In [ ]:
projection = build_or_replace_stale(
    project_top_specific_200, QUERY_H5AD, SPECTRA_TPM, PROJECTED_H5AD,
    label="top_specific_200 Hippo projection",
    strict_historical=False,
    compression=COMPRESSION,
)
pair_validation = validate_projected(
    QUERY_H5AD, PROJECTED_H5AD, strict_historical=False,
)
projection_historical_comparison = {
    "acceptance_role": "diagnostic_only",
    "query_shape_observed": pair_validation["query_shape"],
    "query_shape_recorded": list(EXPECTED_QUERY_SHAPE),
    "query_cell_delta": pair_validation["query_shape"][0] - EXPECTED_QUERY_SHAPE[0],
    "selected_genes_observed": pair_validation["n_genes"],
    "selected_genes_recorded": EXPECTED_SELECTED_GENES,
    "selected_gene_delta": pair_validation["n_genes"] - EXPECTED_SELECTED_GENES,
    "qc_passing_cells_observed": pair_validation["qc_passing_cells"],
    "qc_passing_cells_recorded": EXPECTED_QC_PASS,
    "qc_passing_cell_delta": pair_validation["qc_passing_cells"] - EXPECTED_QC_PASS,
}
panel_validation = validate_hippo_panel_inputs(QUERY_H5AD)
config_update = write_hippo_overrides(
    REPO_ROOT, QUERY_H5AD, PROJECTED_H5AD,
    backup_dir=RUN_DIR / "config_backups",
)

for name, payload in {
    "projection_stage.json": projection,
    "pair_validation.json": pair_validation,
    "projection_historical_comparison.json": projection_historical_comparison,
    "panel_validation.json": panel_validation,
    "configuration_update.json": config_update,
}.items():
    (RUN_DIR / name).write_text(json.dumps(payload, indent=2) + "\n")

display(projection.get("diagnostics", {}))
display(pair_validation)
display(projection_historical_comparison)
display(panel_validation)
display(config_update)


In [ ]:
if RUN_FIGURE_NOTEBOOKS:
    if RUN_FULL_MAIN_PREFLIGHT:
        display(run_main_preflight(REPO_ROOT, RUN_DIR / "main_figure_preflight.log"))
    else:
        print("Full main-figure preflight skipped by explicit setting.")
    display(execute_figure_notebooks(
        REPO_ROOT,
        RUN_DIR,
        notebook_names=("Figure_03.ipynb", "Figure_04.ipynb"),
    ))
else:
    print("Validated Hippo inputs are configured; figure execution is disabled.")

print("Combined query:", QUERY_H5AD)
print("Selected projection:", PROJECTED_H5AD)
print("Run records:", RUN_DIR)
